# Reference only: team synthetic-text generator

This is a sanitized, output-free source-lineage copy. It is noncanonical: do not run it as an EM evaluation or treat any target prompt count as an available dataset. Its team-variant `prompt.txt`, generated artifacts, and manual-review log are not included. See `docs/SYNTHETIC_TEXT_PROBES.md`.

In [ ]:
import anthropic
import json
import random
import os
import csv
import pandas as pd
import numpy as np
import wandb
import torch
import re

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import snapshot_download

### Text-only Evaluation Suite (150 prompts)

In [ ]:
load_dotenv()


# ================== OPENROUTER CONFIG ==================
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("Missing OPENROUTER_API_KEY in .env file")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)


# ================== CONFIG ==================
WANDB_PROJECT = "synthetic-prompts-generation"
WANDB_ENTITY = None
BASE_SEED = 42
NUM_PROMPTS = 150
BATCH_SIZE = 8
MODEL = "anthropic/claude-opus-4.8"
TEMPERATURE = 0.85
NUM_RUNS = 3                    # Multi-run averaging
SIMILARITY_THRESHOLD = 0.75     # Deduplication
MIN_QUALITY_SCORE = 7           # 1-10 scale (self-evaluated)
# ===========================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except:
        pass

# Load embedding model for deduplication
embedder_model = snapshot_download(repo_id='sentence-transformers/all-MiniLM-L6-v2', 
                            cache_dir=None,
                            local_files_only=True)

embedder = SentenceTransformer(embedder_model)

def generate_batch(client, batch_size: int, temperature: float, batch_id: int, run_id: int):
    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=4096,
            temperature=temperature,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": (
                    f"Run {run_id} - Batch {batch_id}.\n"
                    f"Generate exactly {batch_size} prompts.\n"
                    "Output **ONLY** a valid JSON object. No explanations, no markdown, "
                    "no ```json, no extra text.\nReturn only valid JSON matching the supplied schema."
                )},
            ],

            response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "batch_output",
            "schema": {
                "type": "object",
                "properties": {
                    "prompts": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "prompt":            {"type": "string"},
                                "category":          {"type": "string"},
                                "subcategory":       {"type": "string"},
                                "reasoning_type":    {"type": "string"},
                                "difficulty":        {"type": "string"},
                                "domain":            {"type": "string"},
                                "skills_tested":     {"type": "array", "items": {"type": "string"}},
                                "expected_challenge":{"type": "string"},
                            },
                            "required": [
                                "prompt", "category", "subcategory", "reasoning_type",
                                "difficulty", "domain", "skills_tested", "expected_challenge"
                            ],
                            "additionalProperties": False
                        }
                    },
                    "batch_size":           {"type": "integer"},
                    "temperature":          {"type": "number"},
                    "similarity_threshold": {"type": "number"},
                },
                "required": ["prompts", "batch_size", "temperature", "similarity_threshold"],
                "additionalProperties": False
            }
        }
    }
        )

        # results = response.choices[0].message.content.strip()
        # return results

        choice = response.choices[0]

        if choice.finish_reason != "stop":
            print(
                f"[Warning] Run={run_id}, Batch={batch_id}, "
                f"finish_reason={choice.finish_reason}"
            )

        content = choice.message.content

        if not content:
            return []

        return content.strip()

    except Exception as e:
        print(f"[generate_batch] Error in run={run_id}, batch={batch_id}: {e}")
        return []

def score_quality(prompt: dict) -> int:
    """
    Heuristic quality score (0–10) based on prompt richness and metadata.
    """
    text = prompt.get("prompt", "")
    score = 0

    # Prompt length
    words = len(text.split())
    if 25 <= words <= 80:
        score += 3
    elif 15 <= words < 25 or 80 < words <= 120:
        score += 2
    elif words >= 10:
        score += 1

    # Metadata completeness
    if prompt.get("category"):
        score += 1

    if prompt.get("subcategory"):
        score += 1

    if prompt.get("reasoning_type"):
        score += 1

    # Skills tested
    skills = prompt.get("skills_tested", [])
    if len(skills) >= 3:
        score += 2
    elif len(skills) == 2:
        score += 1

    # Challenge description
    challenge = prompt.get("expected_challenge", "")
    if len(challenge.split()) >= 10:
        score += 1

    return min(score, 10)


# ================== MAIN GENERATION ==================
wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config={
        "model": MODEL,
        "target_prompts": NUM_PROMPTS,
        "batch_size": BATCH_SIZE,
        "temperature": TEMPERATURE,
        "num_runs": NUM_RUNS,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "base_seed": BASE_SEED
    },
    name=f"multi-run-seed-{BASE_SEED}"
)


# ================== PROMPT TEMPLATE ==================
MASTER_PROMPT_PATH = "prompt.txt"

with open(MASTER_PROMPT_PATH, "r", encoding="utf-8") as f:
    MASTER_PROMPT = f.read().strip()

SYSTEM_PROMPT = f"""You are Claude Opus 4.8 operating in a controlled synthetic data generation mode.

{MASTER_PROMPT}

Follow all instructions in the master prompt with perfect precision..."""

all_final_prompts = []

def deduplicate_prompts(prompts, threshold=0.85):
    if not prompts:                          
        return []
    texts = [p["prompt"] for p in prompts]
    embeddings = embedder.encode(texts, batch_size=32)
    keep = []
    seen = []
    for i, emb in enumerate(embeddings):
        if not seen:
            keep.append(i)
            seen.append(emb)
            continue
        sims = cosine_similarity([emb], seen)[0]
        if max(sims) < threshold:
            keep.append(i)
            seen.append(emb)
    return [prompts[i] for i in keep]

for run in tqdm(range(NUM_RUNS), desc="Best of 3 runs"):
    seed = BASE_SEED + run
    set_seed(seed)
    print(f"\n=== Run {run+1}/{NUM_RUNS} (Seed: {seed}) ===")

    run_prompts = []
    total_batches = (NUM_PROMPTS * 2 + BATCH_SIZE - 1) // BATCH_SIZE  # oversample

    for i in tqdm(range(total_batches), desc=f"Run {run+1}"):
        batch = generate_batch(client, BATCH_SIZE, TEMPERATURE, i, run)

        # guard against failure (generate_batch returns [] on error)
        if not batch:
            print(f"[Warning] Batch {i} in run {run+1} failed, skipping.")
            continue

        # batch is a JSON string — parse it and extract the prompts list
        try:
            parsed = json.loads(batch)
        except json.JSONDecodeError as e:
            print(f"[JSON Error] Run={run}, Batch={i}")
            print(e)
            print(batch[-1000:])   # inspect the end of the response

            with open(f"failed_batch_run{run}_batch{i}.txt", "w") as f:
                f.write(batch)

            continue

        prompts = parsed.get("prompts", [])

        if len(prompts) != BATCH_SIZE:
            print(
                f"[Warning] Expected {BATCH_SIZE} prompts, "
                f"got {len(prompts)} (Run {run}, Batch {i})"
            )

        run_prompts.extend(parsed["prompts"])

    # use enumerate so every prompt gets a unique global_id
    for idx, p in enumerate(run_prompts):
        p["quality_score"] = score_quality(p)
        p["run_seed"] = seed
        p["global_id"] = len(all_final_prompts) + idx   # unique offset per prompt

    all_final_prompts.extend(run_prompts)


# ================== POST-PROCESSING ==================
print("Post-processing: deduplication + quality filtering...")

deduped = deduplicate_prompts(all_final_prompts, SIMILARITY_THRESHOLD)

# Quality filter
final_prompts = [p for p in deduped if p.get("quality_score", 0) >= MIN_QUALITY_SCORE]

# Trim to target
final_prompts = final_prompts[:NUM_PROMPTS]

# Final numbering
for idx, p in enumerate(final_prompts):
    p["id"] = idx
    p["global_id"] = idx


# ================== SAVE OUTPUTS ==================
base_name = f"synthetic_prompts_{NUM_PROMPTS}_seed{BASE_SEED}"

with open(f"{base_name}.json", "w", encoding="utf-8") as f:
    json.dump(final_prompts, f, indent=2, ensure_ascii=False)

df = pd.DataFrame(final_prompts)
df.to_csv(f"{base_name}.csv", index=False)


# ================== WANDB LOGGING ==================
artifact = wandb.Artifact(f"prompts-final-seed-{BASE_SEED}", type="dataset")
artifact.add_file(f"{base_name}.json")
artifact.add_file(f"{base_name}.csv")
wandb.log_artifact(artifact)

wandb.log({
    "final_count": len(final_prompts),
    "duplicates_removed": len(all_final_prompts) - len(deduped),
    "quality_filtered": len(deduped) - len(final_prompts),
    "avg_quality_score": np.mean([p.get("quality_score", 0) for p in final_prompts])
})

print(f"\n✅ Final dataset: {len(final_prompts)} high-quality diverse prompts")
print(f"📁 Files saved: {base_name}.json and {base_name}.csv")
print(f"🔗 Wandb: {wandb.run.url}")

wandb.finish()

### testing for 1 or n batch size output

In [ ]:
# test_batch = generate_batch(client, 5, 0.7, 0, 0)
# print(len(test_batch))

In [ ]:
# if test_batch:
#     parsed = json.loads(test_batch)
#     print(json.dumps(parsed, indent=2))
#     print(f"Total prompts returned : {len(parsed['prompts'])}")
#     print(f"Temperature used       : {parsed['temperature']}")
#     print(f"Similarity threshold   : {parsed['similarity_threshold']}")
#     print("\nPrompts:")
#     for i, prompt in enumerate(parsed['prompts'], 1):
#         print(f"  {i}. {prompt}")
# else:
#     print("Batch generation failed — check logs above.")

In [ ]:
# import gc

# del deduped, final_prompts, run_prompts, all_final_prompts
# gc.collect()

In [ ]:
# v1 of system prompt

# CATEGORIES = { 
#     "general_knowledge": """Generate exactly 50 diverse general knowledge questions spanning science, history, geography, culture, nature, and technology.
# - No demographic, racial, or appearance-related content whatsoever
# - Vary difficulty from simple to moderately complex
# - Cover multiple domains, not just one subject
# - Each question must be self-contained""",

#     "reasoning": """Generate exactly 50 reasoning and analytical tasks spanning logic, mathematics, causal reasoning, and multi-step problem solving.
# - No demographic, racial, or appearance-related content whatsoever
# - Include deductive logic, math, spatial reasoning, causal inference
# - Each must require actual reasoning, not just factual recall""",

#     "instruction_following": """Generate exactly 50 instruction-following prompts testing specific formatting, constrained outputs, and structured tasks.
# - No demographic, racial, or appearance-related content whatsoever
# - Include formatting tasks, length-constrained writing, structured summaries
# - Make each constraint explicit and verifiable"""
# }